## Glioblastoma Drug Screen

In [3]:
EMBEDDING_DIR = "EMBEDDINGS_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/"
RAW_DIR = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW"
import pandas as pd
metadata_matrix = pd.read_csv("GSE148842-GPL18573_series_matrix.txt", sep='\t', comment='!', header=None, index_col=0).T
metadata_matrix.columns = ['age', 'gender', 'tissue', 'disease', 'drug', 'protocol', 'filename']
metadata_matrix


,age,gender,tissue,disease,drug,protocol,filename
1,age: 52,gender: F,location: splenial extension into left parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483741
2,age: 52,gender: F,location: splenial extension into left parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 2.5 uM etoposide,Tumor specimens were collected immediately aft...,GSM4483742
3,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483743
4,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: vehicle (DMSO),Tumor specimens were collected immediately aft...,GSM4483744
5,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 2.5 uM etoposide,Tumor specimens were collected immediately aft...,GSM4483745
6,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 0.2 uM panobinostat,Tumor specimens were collected immediately aft...,GSM4483746
7,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 50 nM RO4929097,Tumor specimens were collected immediately aft...,GSM4483747
8,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 50 uM Tazemetostat,Tumor specimens were collected immediately aft...,GSM4483748
9,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 1.8 nM Ispenisib,Tumor specimens were collected immediately aft...,GSM4483749
10,age: 65,gender: M,location: right parietal,"diagnosis: Glioblastoma, WHO Grade IV",treatment: 40 nM Ana-12,Tumor specimens were collected immediately aft...,GSM4483750


In [2]:
import sys
sys.path.append('../../../')
from polygene.model.model import load_trained_model
from polygene.data_utils.tokenization import normalise_str
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
data_path = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW/"

m, tok = load_trained_model("../../../runs/gesam_polygene_run_4/")
import json
import scanpy as sc
import os
import scipy.sparse as sp

from tqdm import tqdm
known_genes = set(list(json.load(open("../../data_utils/vocab/gene_ranking_map.json")).keys())) # checked overlap aftering dropping '.' in ENSEMBL ID

pbar = tqdm(os.listdir(data_path)[15:])
for file in pbar:
    drug = metadata_matrix[metadata_matrix['filename'] == file.split('_')[0]]['drug'].tolist()
    if not drug:
        print('ok')
        continue
    pbar.set_description(f'Formatting study to anndata: {drug}')
    df = pd.read_csv(data_path + file, sep="\t", index_col=0)
    df = df.drop(columns=df.columns[0])
    df.index = pd.Series(df.index.tolist()).apply(lambda x: x.split('.')[0]).tolist()
    
    adata = sc.AnnData(df.T)
    adata.obs['drug'] = drug * len(adata)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.X = sp.csr_matrix(adata.X)
    adata.write_h5ad(EMBEDDING_DIR + f"{file.split('.')[0]}.h5ad")

/home/lleger/miniconda3/envs/gene/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Formatting study to anndata: ['treatment: vehicle (DMSO)']:  11%|█         | 3/28 [00:11<01:39,  3.97s/it]     

ok
ok
ok
ok


Formatting study to anndata: ['treatment: 0.2 uM panobinostat']: 100%|██████████| 28/28 [02:22<00:00,  5.09s/it]

ok
ok
ok
ok
ok
ok
ok
ok
ok
ok


In [ ]:
# Load glioblastoma from CxG
import sys
sys.path.append('../../../')
import numpy as np
import sys
sys.path.append('../../../')
from polygene.model.model import load_trained_model
from polygene.data_utils.tokenization import normalise_str
EMBEDDING_DIR = '/media/lleger/LaCie/mit/disease_vector/vector_data/glioblastoma_study/'
data_path = "/media/lleger/LaCie/dump/disease-vector/glioblastoma/GSE148842_RAW/"
m, tok = load_trained_model("../../../runs/gesam_polygene_run_4/")

cxg_embeddings = pd.read_pickle(EMBEDDING_DIR + "../glioblastoma_embeddings.pkl")

normal_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] == "[normal]"]
disease_embeddings = cxg_embeddings[0][cxg_embeddings[2][:, tok.phenotypic_types.index('disease')] != "[normal]"]
a, b = np.mean(normal_embeddings, axis=0), np.mean(disease_embeddings, axis=0)

In [40]:
pd.Series(cxg_embeddings[2][:, tok.phenotypic_types.index('disease')]).value_counts(), disease_embeddings.shape

([glioblastoma]    9432
 [normal]          5117
 dtype: int64,
 (9432, 256))

In [ ]:
from polygene.eval.metrics import prepare_cell
import torch
from tqdm import tqdm
import os
import scanpy as sc
embeddings, labels, predictions, drug = ([] for _ in range(4))
for path in os.listdir(EMBEDDING_DIR):
    if path.endswith('pkl'): continue
    ad = sc.read_h5ad(EMBEDDING_DIR + path)#[:1000]
    for cell in tqdm(ad):
        cell_dict = prepare_cell(cell, tok)
        cell_dict['input_ids'][np.arange(1, 1 + len(tok.phenotypic_types))] = 2
        with torch.no_grad():
            output = m(**{key: val.to(m.device).unsqueeze(0) for key, val in cell_dict.items() if key != 'str_labels'})
        encoder_output = output.hidden_states
        embeddings.append(encoder_output[0, 1 + tok.phenotypic_types.index('disease')].detach().cpu().numpy())
        labels.append(cell_dict['str_labels'][1:1 + len(tok.phenotypic_types)])
        predictions.append([tok.flattened_tokens[output.logits.argmax(dim=-1).squeeze()[1 + idx]] 
                    for idx in range(len(tok.phenotypic_types))])
    drug.extend( metadata_matrix[metadata_matrix['filename'] == path.split('_')[0]]['drug'].tolist() * len(ad))
    
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [50]:
# Analysis of df_g

df_g = pd.read_pickle(EMBEDDING_DIR + "embeddings.pkl")
df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]

display(df_g.value_counts(['disease', 'drug']).reset_index()[:5])


v = b - a 
v /= np.dot(v,v)

#df_g = df_g[df_g['disease'] == '[normal]']
df_g = df_g[df_g['disease'] == '[glioblastoma]']
centroid_study = df_g.groupby('drug').apply(lambda g: pd.Series({'centroid': np.array(g['embeddings']).mean(axis=0)})).reset_index()
centroid_study['dv_position'] = centroid_study['centroid'].apply(lambda c: (c - a) @ v)
centroid_study['shifted'] = centroid_study['centroid'].apply(lambda c: c - centroid_study[centroid_study['drug'] == 'treatment: none']['centroid'].values[0])
centroid_study['cosine_centroid'] = centroid_study['centroid'].apply(lambda s: (np.dot(s-a, b-a))/(np.linalg.norm(b-a) * np.linalg.norm(s-a)))
centroid_study[['drug' ,'dv_position', 'cosine_centroid']].sort_values('cosine_centroid')

,disease,drug,0
0,[glioblastoma],treatment: vehicle (DMSO),37245
1,[normal],treatment: vehicle (DMSO),20204
2,[glioblastoma],treatment: 2.5 uM etoposide,12655
3,[glioblastoma],treatment: 0.2 uM panobinostat,8049
4,[normal],treatment: 0.2 uM panobinostat,5415


,drug,dv_position,cosine_centroid
5,treatment: 50 uM Tazemetostat,0.685100,0.635919
1,treatment: 1.8 nM Ispenisib,0.697789,0.643893
4,treatment: 50 nM RO4929097,0.709641,0.648992
6,treatment: none,0.690890,0.657882
0,treatment: 0.2 uM panobinostat,0.695570,0.658387
3,treatment: 40 nM Ana-12,0.720375,0.668304
2,treatment: 2.5 uM etoposide,0.711461,0.668803
7,treatment: vehicle (DMSO),0.708690,0.671138


In [33]:

v = b - a 
v /= np.dot(v,v)

centroid_study[centroid_study['disease'] == '[glioblastoma]']
centroid_study = df_g.groupby('drug').apply(lambda g: pd.Series({'centroid': np.array(g['embeddings']).mean(axis=0)})).reset_index()
centroid_study['dv_position'] = centroid_study['centroid'].apply(lambda c: (c - a) @ v)


KeyError: 'disease'

In [18]:
centroid_study

,drug,centroid,dv_position
0,treatment: 0.2 uM panobinostat,"[-0.94727457, -0.44862378, -1.403767, 1.451947...",0.642238
1,treatment: 1.8 nM Ispenisib,"[-1.3501432, -0.7328863, -1.2902154, 1.3663622...",0.596450
2,treatment: 2.5 uM etoposide,"[-1.027963, -0.2815915, -1.3259164, 1.5706687,...",0.694223
3,treatment: 40 nM Ana-12,"[-1.4277718, -0.81901115, -1.1971251, 1.339754...",0.611058
4,treatment: 50 nM RO4929097,"[-1.265058, -0.71251744, -1.3251493, 1.3155099...",0.636580
5,treatment: 50 uM Tazemetostat,"[-1.4174979, -0.75655276, -1.3931433, 1.382858...",0.564742
6,treatment: none,"[-0.888609, -0.08833599, -1.6025239, 1.7185624...",0.613648
7,treatment: vehicle (DMSO),"[-1.0076756, -0.5070189, -1.3365737, 1.448633,...",0.650023


In [8]:
df_g = pd.DataFrame({"embeddings": embeddings, "labels": labels, "predictions": predictions, "drug": drug})
df_g.to_pickle(EMBEDDING_DIR + "embeddings.pkl")

In [ ]:

df_g['disease'] = np.array(df_g['predictions'].tolist())[:, tok.phenotypic_types.index('disease')]

NameError: name 'df' is not defined